# Baby Names - the 9 selected sketches, interactive & critique-revised

This notebook implements the 9 sketches kept in `process selection #1`,
revised after an expert multi-agent
critique, and now equipped with:

- name dropdowns over the full ~15,270-name list - 1.13 offers three removable selected-name layers
  to overlay and compare several names at once (plus a free search box); 2.4 offers
  one (the search box overrides it when non-empty);
- date gauges on every chart where it makes sense - year-window sliders (1.5,
  1.13), decade-window sliders (1.2), decade sliders (2.2, 2.6, 3.2, 3.5),
  and the year slider on 3.3.
- sex selectors in English - `Mixed`, `Boys`, `Girls` on 1.5, 1.2, 1.13,
  2.2, and 3.2.

PNG exports show each interaction's default state; HTML exports are fully interactive.


In [ ]:
import altair as alt
import pandas as pd
import numpy as np
import geopandas as gpd  # conda install -c conda-forge geopandas
import json

# Store chart data in external altair-data-*.json files instead of inlining it:
# with ~250k-row tables embedded in every spec the executed notebook reached
# ~50 MB and papermill hit a MemoryError while serialising it. External files
# keep the notebook light; vl-convert still resolves them for the PNG exports.
alt.data_transformers.enable('json')
pass

## Loading and preparing the data


In [ ]:
# Memory-lean load: another heavy job is running on this machine (~2.5 GB free),
# so we read the CSV with categorical dtypes (15k name strings stored once), derive
# the numeric year from the 122 'annais' categories instead of 3.5M strings, then
# relax categories back to plain objects (the strings stay shared) so every later
# groupby keeps its normal semantics.
raw = pd.read_csv('dpt2020.csv', sep=';',
                  dtype={'sexe': 'int8', 'preusuel': 'category',
                         'annais': 'category', 'dpt': 'category', 'nombre': 'int32'})
cat_years = pd.to_numeric(pd.Index(raw['annais'].cat.categories), errors='coerce').astype('float32')
raw['year'] = np.asarray(cat_years)[raw['annais'].cat.codes]
raw.drop(columns=['annais'], inplace=True)
raw['decade'] = (raw['year'] // 10 * 10)
raw['preusuel'] = raw['preusuel'].astype(object)  # shared strings, plain groupbys
raw['dpt'] = raw['dpt'].astype(object)

records = raw[raw.dpt != 'XX'].copy()
named = records[records.preusuel != '_PRENOMS_RARES'].copy()

dept_total = records.groupby('dpt', as_index=False)['nombre'].sum().rename(
    columns={'nombre': 'dept_total'})

ALL_NAMES = sorted(named.preusuel.unique().tolist())  # full dropdown option list
DECADES = list(range(1900, 2021, 10))
# Shared period options for charts that need an all-years aggregate (2.4).
DECADE_OPTS = [0] + DECADES
DECADE_LABELS = ['All years'] + [str(d) for d in DECADES]
SEX_OPTIONS = ['Mixed', 'Boys', 'Girls']
SEX_LABELS = ['mixed', 'boys', 'girls']

import gc
del raw
gc.collect()

print(f'{len(named):,} named rows; {len(ALL_NAMES):,} distinct names.')
named.sample(3)


Department outlines (Corsican `2A`/`2B` merged into the historical code `20`),
and the full per-(department, name) rate table joined to the polygons by a Vega
lookup (geometry stored once, not once per name).


In [ ]:
depts = gpd.read_file('departements-version-simplifiee.geojson')
_corsica = depts['code'].isin(['2A', '2B'])
depts = pd.concat([
    depts[~_corsica],
    gpd.GeoDataFrame({'code': ['20'], 'nom': ['Corse']},
                     geometry=[depts.loc[_corsica, 'geometry'].union_all()],
                     crs=depts.crs),
], ignore_index=True)
METRO = set(depts['code'])

dept_name = named.groupby(['dpt', 'preusuel'], as_index=False)['nombre'].sum()
dept_name = dept_name.merge(dept_total, on='dpt')
dept_name['per_1000'] = (1000 * dept_name['nombre'] / dept_name['dept_total']).round(3)
dept_name = dept_name.merge(depts[['code', 'nom']], left_on='dpt', right_on='code',
                            how='left').drop(columns=['code', 'dept_total'])

geo_features = alt.InlineData(values=json.loads(depts.to_json()),
                              format=alt.DataFormat(property='features'))
FRANCE_PROJECTION = dict(type='conicConformal', rotate=[-3, 0],
                         center=[0, 46.5], parallels=[44, 49])
print(f'{len(dept_name):,} (department, name) pairs embedded.')
dept_name.sample(3)

# Visualization 1 - How do names evolve over time?


### Sketch 1.5 - "Volume" streamgraph (Top-N gauge, ranked on the chosen period)

Centred stacked bands. The Top N slider picks how many names to show, and the
ranking is recomputed over the selected From/To window (e.g. 1900-1930 surfaces
MARIE/JEAN/LOUIS; 1990-2020 surfaces LÉA/THOMAS...). Names are written on the bands
at their in-window peak; click a band to isolate it.

*Implementation note:* the ranking must happen client-side (it depends on the
window), so the chart uses Vega-Lite's native centred stacking plus a
`joinaggregate` total + `dense_rank` window transform over the full per-year series
of all ~15k names.


In [ ]:
# Full (year, name, sex mode) series. The Top-N ranking is computed client-side
# on the selected window, so every name must be available to the chart.
_stream_base = named.dropna(subset=['year'])
stream_mixte = (_stream_base.groupby(['year', 'preusuel'], as_index=False)['nombre'].sum())
stream_mixte['sex_mode'] = 'Mixed'
stream_sex = (_stream_base.groupby(['year', 'preusuel', 'sexe'], as_index=False)['nombre'].sum())
stream_sex['sex_mode'] = np.where(stream_sex['sexe'].eq(1), 'Boys', 'Girls')
stream_src = pd.concat([
    stream_mixte[['year', 'preusuel', 'nombre', 'sex_mode']],
    stream_sex[['year', 'preusuel', 'nombre', 'sex_mode']],
], ignore_index=True)
stream_src['year'] = stream_src['year'].astype(int)

y_from = alt.param(name='y_from', value=1900,
                   bind=alt.binding_range(min=1900, max=2020, step=1, name='From year  '))
y_to = alt.param(name='y_to', value=2020,
                 bind=alt.binding_range(min=1900, max=2020, step=1, name='To year  '))
n_top = alt.param(name='n_top', value=8,
                  bind=alt.binding_range(min=3, max=50, step=1,
                                         name='Top N names (ranked on the window)  '))
sex_15 = alt.param(name='sex_15', value='Mixed',
                   bind=alt.binding_select(options=SEX_OPTIONS, labels=SEX_LABELS,
                                           name='Sex  '))

isolate = alt.selection_point(fields=['preusuel'], on='click', clear='dblclick', empty=True)

# Shared pipeline: sex filter -> window filter -> per-name total on the window ->
# dense rank -> keep the Top N -> per-name in-window peak (for the labels).
def piped(chart):
    return (chart
            .transform_filter('datum.sex_mode == sex_15')
            .transform_filter('datum.year >= y_from && datum.year <= y_to')
            .transform_joinaggregate(tot='sum(nombre)', groupby=['preusuel'])
            .transform_window(rk='dense_rank()',
                              sort=[alt.SortField('tot', order='descending')])
            .transform_filter('datum.rk <= n_top')
            .transform_joinaggregate(maxn='max(nombre)', groupby=['preusuel']))

bands = piped(alt.Chart(stream_src)).mark_area(interpolate='monotone').encode(
    x=alt.X('year:Q', title='Year', axis=alt.Axis(format='d', grid=False)),
    y=alt.Y('nombre:Q', stack='center', title=None, axis=None),
    color=alt.Color('preusuel:N', legend=None, scale=alt.Scale(scheme='tableau20')),
    order=alt.Order('tot:Q', sort='descending'),
    opacity=alt.condition(isolate, alt.value(0.9), alt.value(0.18)),
    tooltip=[alt.Tooltip('preusuel:N', title='Name'),
             alt.Tooltip('sex_mode:N', title='Sex mode'),
             alt.Tooltip('year:Q', title='Year', format='d'),
             alt.Tooltip('nombre:Q', title='Births', format=',')],
)
# Label layers share the SAME stacked pipeline (so they land on the bands) and only
# print text on each band's in-window peak row.
label_text = alt.condition('datum.nombre == datum.maxn',
                           alt.Text('preusuel:N'), alt.value(''))
halo = piped(alt.Chart(stream_src)).mark_text(
    baseline='top', dy=2, fontSize=11, fontWeight='bold',
    stroke='white', strokeWidth=3).encode(
    x='year:Q', y=alt.Y('nombre:Q', stack='center', axis=None),
    detail='preusuel:N', order=alt.Order('tot:Q', sort='descending'), text=label_text)
labels = piped(alt.Chart(stream_src)).mark_text(
    baseline='top', dy=2, fontSize=11, fontWeight='bold', color='#111').encode(
    x='year:Q', y=alt.Y('nombre:Q', stack='center', axis=None),
    detail='preusuel:N', order=alt.Order('tot:Q', sort='descending'), text=label_text)

sketch_1_5 = (bands + halo + labels).add_params(isolate, y_from, y_to, n_top, sex_15).properties(
    width=820, height=380,
    title='1.5 - Streamgraph of the Top-N names on the chosen window (drag N, years, and sex)')

sketch_1_5.save('sketch_1_5_streamgraph.png', ppi=120)
sketch_1_5


### Sketch 1.2 - Bump-chart leaderboard (decade-window gauges)

Per-decade rank (1 = top); only names that ever reached the top 3 are coloured and
labelled, the rest are grey context. Two sliders bound the range of decades
shown; hover to highlight.


In [ ]:
_dec_base = named[named.year.isin(DECADES)]
dec_mixte = (_dec_base.groupby(['year', 'preusuel'], as_index=False)['nombre'].sum())
dec_mixte['sex_mode'] = 'Mixed'
dec_sex = (_dec_base.groupby(['year', 'preusuel', 'sexe'], as_index=False)['nombre'].sum())
dec_sex['sex_mode'] = np.where(dec_sex['sexe'].eq(1), 'Boys', 'Girls')
dec = pd.concat([
    dec_mixte[['year', 'preusuel', 'nombre', 'sex_mode']],
    dec_sex[['year', 'preusuel', 'nombre', 'sex_mode']],
], ignore_index=True)
dec['rank'] = dec.groupby(['year', 'sex_mode'])['nombre'].rank(ascending=False, method='first')
dec = dec[dec['rank'] <= 40].sort_values(['sex_mode', 'preusuel', 'year']).copy()
dec['run'] = dec.groupby(['sex_mode', 'preusuel'])['year'].transform(lambda s: (s.diff() > 10).cumsum())
# Single-point line groups trigger tiny-skia path-stroking warnings in PNG export.
dec['run_len'] = dec.groupby(['sex_mode', 'preusuel', 'run'])['year'].transform('size')
featured = set(dec.loc[dec['rank'] <= 3, 'preusuel'])
dec['featured'] = dec['preusuel'].isin(featured)
feat, ctx = dec[dec.featured], dec[~dec.featured]
# Anchor each label at the name's last appearance IN THE TOP-8 for the current sex mode.
feat8 = feat[feat['rank'] <= 8]
last_seen = feat8.loc[feat8.groupby(['sex_mode', 'preusuel'])['year'].idxmax()].sort_values(['sex_mode', 'rank', 'year']).copy()
last_seen['k'] = last_seen.groupby(['sex_mode', 'rank']).cumcount()
last_seen['rank_lab'] = last_seen['rank'] + (last_seen['k'] % 2) * 0.34 - 0.17

d_from = alt.param(name='d_from', value=1900,
                   bind=alt.binding_range(min=1900, max=2020, step=10, name='From decade  '))
d_to = alt.param(name='d_to', value=2020,
                 bind=alt.binding_range(min=1900, max=2020, step=10, name='To decade  '))
sex_12 = alt.param(name='sex_12', value='Mixed',
                   bind=alt.binding_select(options=SEX_OPTIONS, labels=SEX_LABELS,
                                           name='Sex  '))
in_window = 'datum.sex_mode == sex_12 && datum.year >= d_from && datum.year <= d_to && datum.rank <= k_rank'

k_rank = alt.param(name='k_rank', value=8,
                   bind=alt.binding_range(min=3, max=40, step=1, name='Ranks shown (top K)  '))
hover = alt.selection_point(fields=['preusuel'], on='pointerover', empty=False)
# Auto-fitting reversed scale: the y domain follows the chosen K (1 stays on top).
y_enc = alt.Y('rank:Q', title='Rank (1 = most common)',
              scale=alt.Scale(reverse=True, zero=False, nice=False, padding=10),
              axis=alt.Axis(values=list(range(1, 41))))
ctx_lines = alt.Chart(ctx).transform_filter(in_window).transform_filter('datum.run_len >= 2').mark_line(
    color='#dcdcdc', strokeWidth=1.1).encode(
    x=alt.X('year:O', title='Decade'), y=y_enc, detail=['sex_mode:N', 'preusuel:N', 'run:N'])
feat_lines = alt.Chart(feat).transform_filter(in_window).transform_filter('datum.run_len >= 2').mark_line(
    point=True, strokeWidth=2.6).encode(
    x=alt.X('year:O', title='Decade'), y=y_enc,
    color=alt.Color('preusuel:N', legend=None, scale=alt.Scale(scheme='category20')),
    detail=['sex_mode:N', 'preusuel:N', 'run:N'],
    opacity=alt.condition(hover, alt.value(1), alt.value(0.9)),
    tooltip=[alt.Tooltip('preusuel:N', title='Name'), alt.Tooltip('sex_mode:N', title='Sex mode'),
             alt.Tooltip('year:O', title='Decade'), alt.Tooltip('rank:Q', title='Rank'),
             alt.Tooltip('nombre:Q', title='Births', format=',')])
feat_labels = alt.Chart(last_seen).transform_filter(in_window).mark_text(
    align='left', dx=8, fontSize=9, fontWeight='bold').encode(
    x='year:O', y='rank_lab:Q', text='preusuel:N',
    color=alt.Color('preusuel:N', legend=None, scale=alt.Scale(scheme='category20')))
sketch_1_2 = (ctx_lines + feat_lines + feat_labels).add_params(hover, d_from, d_to, k_rank, sex_12).properties(
    width=760, height=430,
    title='1.2 - Leaderboard by decade - names that ever reached the top 3 (choose sex)')

sketch_1_2.save('sketch_1_2_leaderboard.png', ppi=120)
sketch_1_2


### Sketch 1.13 - Compare names: dropdown slots, search, or Top-K of the window

Two ways to choose what is drawn over the grey top-30 context:

- pick names - up to 20 removable dropdown slots spanning all ~15,270 names. All slots default to `(none)`, and the Name slots shown slider controls how many numbered slots are visible and allowed to draw;
- or take the Top-K of the window - the K slider draws the K most popular names *of the current From/To window* as thin coloured curves (K = 0 switches this off).

Two sliders set the year window; the ranking follows it. The sex selector recomputes the Top-K for mixed / boys / girls.


In [ ]:
_ts_base = named.dropna(subset=['year'])
ts_mixte = (_ts_base.groupby(['year', 'preusuel'], as_index=False)['nombre'].sum())
ts_mixte['sex_mode'] = 'Mixed'
ts_sex = (_ts_base.groupby(['year', 'preusuel', 'sexe'], as_index=False)['nombre'].sum())
ts_sex['sex_mode'] = np.where(ts_sex['sexe'].eq(1), 'Boys', 'Girls')
ts_all = pd.concat([
    ts_mixte[['year', 'preusuel', 'nombre', 'sex_mode']],
    ts_sex[['year', 'preusuel', 'nombre', 'sex_mode']],
], ignore_index=True)

top30_by_mode = (ts_all.groupby(['sex_mode', 'preusuel'], as_index=False)['nombre'].sum()
                 .sort_values(['sex_mode', 'nombre'], ascending=[True, False])
                 .groupby('sex_mode').head(30)[['sex_mode', 'preusuel']])
ts_ctx = ts_all.merge(top30_by_mode, on=['sex_mode', 'preusuel'])

NAME_PICK_OPTIONS = ['__NONE__'] + ALL_NAMES
NAME_PICK_LABELS = ['(none)'] + ALL_NAMES
NAME_SLOT_COUNT = 20
NAME_SLOT_COLORS = [
    '#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd',
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
    '#b2182b', '#2166ac', '#1b7837', '#b35806', '#542788',
    '#a6611a', '#c51b7d', '#4d4d4d', '#7fbc41', '#35978f',
]
name_slots = alt.param(name='name_slots', value=3,
                       bind=alt.binding_range(min=0, max=NAME_SLOT_COUNT, step=1,
                                              name='Name slots shown  '))
name_param_defs = []
for i in range(1, NAME_SLOT_COUNT + 1):
    param_name = f'name_{i:02d}'
    param_obj = alt.param(
        name=param_name,
        value='__NONE__',
        bind=alt.binding_select(options=NAME_PICK_OPTIONS, labels=NAME_PICK_LABELS,
                                name=f'Name {i:02d}  '),
    )
    name_param_defs.append((i, param_name, param_obj))

name_q = alt.param(name='name_q', value='',
                   bind=alt.binding(input='search', name='Free search  ',
                                    placeholder='e.g. KEVIN'))
sex_113 = alt.param(name='sex_113', value='Mixed',
                    bind=alt.binding_select(options=SEX_OPTIONS, labels=SEX_LABELS,
                                            name='Sex  '))
k_sel = alt.param(name='k_sel', value=8,
                  bind=alt.binding_range(min=0, max=30, step=1,
                                         name='Top-K of the window (0 = off)  '))
w_from = alt.param(name='w_from', value=1900,
                   bind=alt.binding_range(min=1900, max=2020, step=1, name='From year  '))
w_to = alt.param(name='w_to', value=2020,
                 bind=alt.binding_range(min=1900, max=2020, step=1, name='To year  '))
in_window = 'datum.sex_mode == sex_113 && datum.year >= w_from && datum.year <= w_to'

context = alt.Chart(ts_ctx).transform_filter(in_window).mark_line(
    color='#c7c7c7', strokeWidth=0.8, opacity=0.4).encode(
    x=alt.X('year:Q', title='Year', axis=alt.Axis(format='d', tickCount=13)),
    y=alt.Y('nombre:Q', title='Births'), detail='preusuel:N')
wars_df = pd.DataFrame({'year': [1914, 1939], 'lab': ['WWI', 'WWII'],
                        'yy': [ts_ctx['nombre'].max() * 0.98] * 2})
wars = alt.Chart(wars_df).transform_filter('datum.year >= w_from && datum.year <= w_to').mark_rule(
    color='#bbb', strokeDash=[2, 2]).encode(x='year:Q')
wars_lab = alt.Chart(wars_df).transform_filter('datum.year >= w_from && datum.year <= w_to').mark_text(
    color='#777', fontSize=9, dx=3, align='left', baseline='top').encode(
    x='year:Q', y='yy:Q', text='lab:N')

def picked_layer(slot_idx, param_name, colour):
    dy = ((slot_idx - 1) % 5 - 2) * 7
    flt = (f"name_slots >= {slot_idx} && {param_name} != '__NONE__' "
           f"&& datum.preusuel == {param_name} && ({in_window})")
    line = alt.Chart(ts_all).transform_filter(flt).mark_line(strokeWidth=2.5, color=colour).encode(
        x='year:Q', y='nombre:Q',
        tooltip=[alt.Tooltip('preusuel:N', title='Name'),
                 alt.Tooltip('sex_mode:N', title='Sex mode'),
                 alt.Tooltip('year:Q', title='Year', format='d'),
                 alt.Tooltip('nombre:Q', title='Births', format=',')])
    # Label each selected series at its in-window peak; end labels collapse for old names.
    tip = (alt.Chart(ts_all).transform_filter(flt)
           .transform_joinaggregate(maxn='max(nombre)')
           .transform_filter('datum.nombre == datum.maxn')
           .mark_text(align='left', dx=6, dy=dy, fontWeight='bold', color=colour, fontSize=11)
           .encode(x='year:Q', y='nombre:Q', text='preusuel:N'))
    return line + tip

manual_layers = [
    picked_layer(slot_idx, param_name, NAME_SLOT_COLORS[(slot_idx - 1) % len(NAME_SLOT_COLORS)])
    for slot_idx, param_name, _ in name_param_defs
]

topk = (alt.Chart(ts_all).transform_filter(in_window)
        .transform_joinaggregate(tot='sum(nombre)', groupby=['preusuel'])
        .transform_window(rk='dense_rank()', sort=[alt.SortField('tot', order='descending')])
        .transform_filter('datum.rk <= k_sel')
        .mark_line(strokeWidth=1.3, opacity=0.85).encode(
            x='year:Q', y='nombre:Q',
            color=alt.Color('preusuel:N', legend=None, scale=alt.Scale(scheme='tableau20')),
            tooltip=[alt.Tooltip('preusuel:N', title='Name'),
                     alt.Tooltip('sex_mode:N', title='Sex mode'),
                     alt.Tooltip('year:Q', title='Year', format='d'),
                     alt.Tooltip('nombre:Q', title='Births', format=',')]))
flt_q = f"name_q != '' && upper(datum.preusuel) == upper(name_q) && ({in_window})"
lay_q = (alt.Chart(ts_all).transform_filter(flt_q).mark_line(strokeWidth=2.8, color='#9467bd')
         .encode(x='year:Q', y='nombre:Q',
                 tooltip=[alt.Tooltip('preusuel:N', title='Name'),
                          alt.Tooltip('sex_mode:N', title='Sex mode'),
                          alt.Tooltip('year:Q', title='Year', format='d'),
                          alt.Tooltip('nombre:Q', title='Births', format=',')]))

sketch_1_13 = alt.layer(context, wars, wars_lab, topk, *manual_layers, lay_q).add_params(
    name_slots, *[param_obj for _, _, param_obj in name_param_defs],
    name_q, sex_113, k_sel, w_from, w_to).properties(
    width=820, height=380,
    title='1.13 - Pick up to 20 names or take the Top-K of the window')

sketch_1_13.save('sketch_1_13_slices.png', ppi=120)
sketch_1_13


# Visualization 2 - Is there a regional effect?


### Sketch 2.4 - Map of France: dropdown over the full 15k list (+ search, period gauge)

Pick any of the ~15,270 names from the dropdown (type-ahead works in the
browser); typing in the search box overrides the dropdown. The period is a
sliding gauge: drag the slider through the century (decade steps - the honest
granularity of the underlying ~980k-row department x name x decade table, served as
an external JSON referenced by URL); tick All years to aggregate 1900-2020.
By default, All years is unchecked and the chart opens on the selected decade.
The active name is printed on the map; hovering outlines a department in orange.


In [ ]:
# (department, name, decade) rate table -> external JSON referenced by URL.
# ~745k rows: kept OUT of the spec (the json transformer would re-serialise it per
# layer); pandas writes it once and both layers point at the same file.
_slim4 = records.loc[records['decade'].notna(), ['dpt', 'decade', 'nombre']]
dec_tot4 = (_slim4.groupby(['dpt', 'decade'], as_index=False)['nombre'].sum()
            .rename(columns={'nombre': 'tot'}))
dn_dec = (named.dropna(subset=['decade'])
          .groupby(['dpt', 'preusuel', 'decade'], as_index=False)['nombre'].sum()
          .merge(dec_tot4, on=['dpt', 'decade']))
dn_dec['per_1000'] = (1000 * dn_dec['nombre'] / dn_dec['tot']).round(2)
dn_dec = dn_dec.drop(columns=['tot'])
dn_all = dept_name.copy()
dn_all['decade'] = 0
dn_dec = pd.concat([dn_dec[['dpt', 'preusuel', 'decade', 'nombre', 'per_1000']],
                    dn_all[['dpt', 'preusuel', 'decade', 'nombre', 'per_1000']]],
                   ignore_index=True)
dn_dec['decade'] = dn_dec['decade'].astype(int)
dn_dec = dn_dec.merge(depts[['code', 'nom']], left_on='dpt', right_on='code',
                      how='left').drop(columns='code')
DN_URL = 'altair_dn_decade.json'
dn_dec.to_json(DN_URL, orient='records')
print(f'{len(dn_dec):,} rows written to {DN_URL}')
del _slim4, dn_all

name_dd = alt.param(name='name_dd', value='JEAN',
                    bind=alt.binding_select(options=ALL_NAMES, name='Name (dropdown)  '))
name_q2 = alt.param(name='name_q2', value='',
                    bind=alt.binding(input='search', name='Search (overrides)  ',
                                     placeholder='e.g. LUCIEN'))
all_yrs = alt.param(name='all_yrs', value=False,
                    bind=alt.binding_checkbox(name='All years (1900-2020)  '))
decade_g = alt.param(name='decade_g', value=1980,
                     bind=alt.binding_range(min=1900, max=2020, step=10,
                                            name='Decade slider  '))
picked = ("(name_q2 != '' ? upper(datum.preusuel) == upper(name_q2)"
          " : datum.preusuel == name_dd)"
          " && (all_yrs ? datum.decade == 0 : datum.decade == decade_g)")
dept_hover = alt.selection_point(fields=['nom'], on='pointerover', clear='pointerout', empty=False)

base_map = alt.Chart(geo_features).mark_geoshape(
    fill='#efefef', stroke='white', strokeWidth=0.4).project(**FRANCE_PROJECTION)
choro = alt.Chart(DN_URL).transform_filter(picked).transform_lookup(
    lookup='dpt', from_=alt.LookupData(geo_features, key='properties.code', fields=['type', 'geometry']),
).mark_geoshape().encode(
    color=alt.Color('per_1000:Q', scale=alt.Scale(scheme='viridis'),
                    legend=alt.Legend(title=['Births', 'per 1000'])),
    stroke=alt.condition(dept_hover, alt.value('#ff7f0e'), alt.value('white')),
    strokeWidth=alt.condition(dept_hover, alt.value(2.5), alt.value(0.4)),
    tooltip=[alt.Tooltip('nom:N', title='Department'), alt.Tooltip('preusuel:N', title='Name'),
             alt.Tooltip('nombre:Q', title='Births', format=','),
             alt.Tooltip('per_1000:Q', title='Births / 1000', format='.2f')],
).project(**FRANCE_PROJECTION)
name_tag = (alt.Chart(DN_URL).transform_filter(picked)
            .transform_aggregate(n='count()', groupby=['preusuel'])
            .transform_calculate(lon='-4.6', lat='51.3')
            .mark_text(align='left', fontSize=17, fontWeight='bold', color='#222')
            .encode(longitude='lon:Q', latitude='lat:Q', text='preusuel:N').project(**FRANCE_PROJECTION))
sketch_2_4 = (base_map + choro + name_tag).add_params(name_dd, name_q2, all_yrs, decade_g, dept_hover).properties(
    width=560, height=520,
    title='2.4 - Where is a name most common? (any name; slide the decade)')

# PNG snapshot: vl-convert cannot fetch URL data (it inlines DataFrames instead),
# so the static export is rendered from the default state (JEAN, 1980) as a
# small in-memory slice; the displayed chart above stays URL-backed + interactive.
_png_slice = dn_dec[(dn_dec.preusuel == 'JEAN') & (dn_dec.decade == 1980)]
_png_choro = alt.Chart(_png_slice).transform_lookup(
    lookup='dpt', from_=alt.LookupData(geo_features, key='properties.code', fields=['type', 'geometry']),
).mark_geoshape(stroke='white', strokeWidth=0.4).encode(
    color=alt.Color('per_1000:Q', scale=alt.Scale(scheme='viridis'),
                    legend=alt.Legend(title=['Births', 'per 1000'])),
).project(**FRANCE_PROJECTION)
_png_tag = alt.Chart(pd.DataFrame({'lon': [-4.6], 'lat': [51.3], 't': ['JEAN']})).mark_text(
    align='left', fontSize=17, fontWeight='bold', color='#222').encode(
    longitude='lon:Q', latitude='lat:Q', text='t:N').project(**FRANCE_PROJECTION)
(base_map + _png_choro + _png_tag).properties(
    width=560, height=520,
    title='2.4 - Where is a name most common? (any name; slide the decade)'
).save('sketch_2_4_map_select.png', ppi=120)
sketch_2_4


### Sketch 2.6 - Small-multiple maps, one per name (decade slider + map-count slider)

Choropleths with independent scales and a shared hover. Each map slot has its own name dropdown.
The decade slider lets you watch each name's stronghold move through the century,
and the Maps shown slider controls how many name dropdowns are visible and active, from 3 to 6.


In [ ]:
map_name_1 = alt.param(name='map_name_1', value='LUCIEN',
                       bind=alt.binding_select(options=ALL_NAMES, name='Name 01  '))
map_name_2 = alt.param(name='map_name_2', value='LEA',
                       bind=alt.binding_select(options=ALL_NAMES, name='Name 02  '))
map_name_3 = alt.param(name='map_name_3', value='GABRIEL',
                       bind=alt.binding_select(options=ALL_NAMES, name='Name 03  '))
map_name_4 = alt.param(name='map_name_4', value='NICOLAS',
                       bind=alt.binding_select(options=ALL_NAMES, name='Name 04  '))
map_name_5 = alt.param(name='map_name_5', value='EMMA',
                       bind=alt.binding_select(options=ALL_NAMES, name='Name 05  '))
map_name_6 = alt.param(name='map_name_6', value='LOUIS',
                       bind=alt.binding_select(options=ALL_NAMES, name='Name 06  '))
n_maps_26 = alt.param(name='n_maps_26', value=6,
                      bind=alt.binding_range(min=3, max=6, step=1, name='Maps shown  '))
decade_26 = alt.param(name='decade_26', value=2010,
                       bind=alt.binding_range(min=1900, max=2020, step=10,
                                              name='Decade  '))
mm_hover = alt.selection_point(fields=['nom'], on='pointerover', clear='pointerout', empty=False)

# The slider controls which map slots contain data; the executed notebook also
# hides dropdown controls above the selected count after papermill renders HTML.
def one_map(param_name, scheme, slot_label, slot_idx):
    filt = (f"n_maps_26 >= {slot_idx} && datum.preusuel == {param_name} "
            "&& datum.decade == decade_26 && isValid(datum.nom)")
    base = alt.Chart(DN_URL).transform_filter(filt).transform_lookup(
        lookup='dpt', from_=alt.LookupData(geo_features, key='properties.code', fields=['type', 'geometry'])
    )
    choro = base.mark_geoshape().encode(
        color=alt.Color('per_1000:Q', scale=alt.Scale(scheme=scheme),
                        legend=alt.Legend(title='/1000', orient='bottom', format='.1f')),
        stroke=alt.condition(mm_hover, alt.value('#111827'), alt.value('#cccccc')),
        strokeWidth=alt.condition(mm_hover, alt.value(1.8), alt.value(0.3)),
        tooltip=[alt.Tooltip('nom:N', title='Department'),
                 alt.Tooltip('preusuel:N', title='Name'),
                 alt.Tooltip('per_1000:Q', title='Births / 1000', format='.2f')],
    ).project(**FRANCE_PROJECTION)
    tag = alt.Chart(pd.DataFrame({'lon': [-4.6], 'lat': [51.3]})).transform_filter(
        f'n_maps_26 >= {slot_idx}'
    ).transform_calculate(
        label=param_name
    ).mark_text(align='left', fontSize=14, fontWeight='bold', color='#222').encode(
        longitude='lon:Q', latitude='lat:Q', text='label:N').project(**FRANCE_PROJECTION)
    return (choro + tag).properties(width=220, height=220, title=slot_label)

schemes = ['greens', 'reds', 'oranges', 'blues', 'purples', 'greys']
slot_defs = [
    ('map_name_1', schemes[0], 'Map 1', 1),
    ('map_name_2', schemes[1], 'Map 2', 2),
    ('map_name_3', schemes[2], 'Map 3', 3),
    ('map_name_4', schemes[3], 'Map 4', 4),
    ('map_name_5', schemes[4], 'Map 5', 5),
    ('map_name_6', schemes[5], 'Map 6', 6),
]
row_26_top = alt.hconcat(*[one_map(*slot_defs[i]) for i in range(3)]).resolve_scale(color='independent')
row_26_bottom = alt.hconcat(*[one_map(*slot_defs[i]) for i in range(3, 6)]).resolve_scale(color='independent')
sketch_2_6 = alt.vconcat(row_26_top, row_26_bottom).resolve_scale(color='independent').add_params(
    map_name_1, map_name_2, map_name_3, map_name_4, map_name_5, map_name_6,
    n_maps_26, mm_hover, decade_26,
).properties(
    title='2.6 - Regional rate of 3 to 6 selectable names (independent scales; slide decade and map count)')

# PNG snapshot with the default six names. The displayed chart above remains fully
# interactive, while the static export avoids URL fetch issues in vl-convert.
def one_map_png(nm, scheme):
    g = depts.merge(dn_dec[(dn_dec.preusuel == nm) & (dn_dec.decade == 2010)], how='right',
                    left_on='code', right_on='dpt').dropna(subset=['geometry'])
    return alt.Chart(g).mark_geoshape(stroke='#cccccc', strokeWidth=0.3).encode(
        color=alt.Color('per_1000:Q', scale=alt.Scale(scheme=scheme),
                        legend=alt.Legend(title='/1000', orient='bottom', format='.1f')),
        tooltip=[alt.Tooltip('nom:N', title='Department'),
                 alt.Tooltip('per_1000:Q', title='Births / 1000', format='.2f')],
    ).project(**FRANCE_PROJECTION).properties(width=220, height=220, title=nm)

row_26_png_top = alt.hconcat(
    one_map_png('LUCIEN', schemes[0]),
    one_map_png('LEA', schemes[1]),
    one_map_png('GABRIEL', schemes[2]),
).resolve_scale(color='independent')
row_26_png_bottom = alt.hconcat(
    one_map_png('NICOLAS', schemes[3]),
    one_map_png('EMMA', schemes[4]),
    one_map_png('LOUIS', schemes[5]),
).resolve_scale(color='independent')
sketch_2_6_png = alt.vconcat(row_26_png_top, row_26_png_bottom).resolve_scale(color='independent').properties(
    title='2.6 - Regional rate of six selectable names (2010 default PNG snapshot)')

sketch_2_6_png.save('sketch_2_6_small_multiples.png', ppi=120)
sketch_2_6


### Sketch 2.2 - Name x department heat-map, row-relative (decade slider + sex selector)

Each cell shows the name's rate relative to its own row maximum for the chosen
period. Slide the decade and switch Mixed / Boys / Girls to recompute both the
ranking and the row-relative regional profile.


In [ ]:
_heat_rank_base = named.dropna(subset=['decade'])
heat_rank_mixed = (_heat_rank_base.groupby(['decade', 'preusuel'], as_index=False)['nombre'].sum()
                   .sort_values(['decade', 'nombre'], ascending=[True, False])
                   .groupby('decade').head(300))
heat_rank_mixed['sex_mode'] = 'Mixed'
heat_rank_sex = (_heat_rank_base.groupby(['decade', 'sexe', 'preusuel'], as_index=False)['nombre'].sum()
                 .sort_values(['decade', 'sexe', 'nombre'], ascending=[True, True, False])
                 .groupby(['decade', 'sexe']).head(300))
heat_rank_sex['sex_mode'] = np.where(heat_rank_sex['sexe'].eq(1), 'Boys', 'Girls')
heat_rank = pd.concat([
    heat_rank_mixed[['decade', 'sex_mode', 'preusuel', 'nombre']],
    heat_rank_sex[['decade', 'sex_mode', 'preusuel', 'nombre']],
], ignore_index=True)
heat_rank['decade'] = heat_rank['decade'].astype(int)
heat_rank['nrank'] = heat_rank.groupby(['decade', 'sex_mode'])['nombre'].rank(ascending=False, method='first').astype(int)
heat_names = heat_rank['preusuel'].drop_duplicates().tolist()
heat_rank = heat_rank[['decade', 'sex_mode', 'preusuel', 'nrank']]
hm = named[named.preusuel.isin(heat_names) & named.dpt.isin(METRO)].copy()
# Slim 4-column projection before filtering: a full-frame dropna() copy of the
# 3.5M-row records table raised MemoryError on this RAM-tight machine.
_slim = records.loc[records['decade'].notna() & records['dpt'].isin(METRO),
                    ['dpt', 'decade', 'sexe', 'nombre']]
dec_tot = (_slim.groupby(['dpt', 'decade'], as_index=False)['nombre'].sum()
           .rename(columns={'nombre': 'tot'}))
dec_tot_sex = (_slim.groupby(['dpt', 'decade', 'sexe'], as_index=False)['nombre'].sum()
           .rename(columns={'nombre': 'tot'}))
del _slim
per_dec_mixed = (hm.dropna(subset=['decade'])
           .groupby(['decade', 'dpt', 'preusuel'], as_index=False)['nombre'].sum()
           .merge(dec_tot, on=['dpt', 'decade']))
per_dec_mixed['sex_mode'] = 'Mixed'
per_dec_sex = (hm.dropna(subset=['decade'])
               .groupby(['decade', 'dpt', 'preusuel', 'sexe'], as_index=False)['nombre'].sum()
               .merge(dec_tot_sex, on=['dpt', 'decade', 'sexe']))
per_dec_sex['sex_mode'] = np.where(per_dec_sex['sexe'].eq(1), 'Boys', 'Girls')
per_dec = pd.concat([per_dec_mixed, per_dec_sex], ignore_index=True)
per_dec['per_1000'] = 1000 * per_dec['nombre'] / per_dec['tot']
heat_df = per_dec[['decade', 'dpt', 'preusuel', 'sex_mode', 'per_1000']].copy()
heat_df['rel'] = (heat_df.groupby(['decade', 'sex_mode', 'preusuel'])['per_1000']
                  .transform(lambda s: s / s.max())).round(3)
heat_df['per_1000'] = heat_df['per_1000'].round(3)
heat_df['decade'] = heat_df['decade'].astype(int)
heat_df = heat_df.merge(heat_rank, on=['decade', 'sex_mode', 'preusuel'])
heat_grid = (heat_rank.assign(_k=1)
             .merge(pd.DataFrame({'dpt': sorted(METRO), '_k': 1}), on='_k')
             .drop(columns='_k'))
heat_df = heat_grid.merge(heat_df, how='left', on=['decade', 'sex_mode', 'preusuel', 'nrank', 'dpt'])
heat_df[['per_1000', 'rel']] = heat_df[['per_1000', 'rel']].fillna(0)
heat_df = heat_df.merge(depts[['code', 'nom']], left_on='dpt', right_on='code').drop(columns='code')
dept_order = [d for d in dept_total.sort_values('dept_total', ascending=False)['dpt'] if d in METRO]

n_rows = alt.param(name='n_rows', value=15,
                   bind=alt.binding_range(min=5, max=300, step=1, name='Top N names (rows)  '))
decade_22 = alt.param(name='decade_22', value=2010,
                       bind=alt.binding_range(min=1900, max=2020, step=10,
                                              name='Decade  '))
sex_22 = alt.param(name='sex_22', value='Mixed',
                   bind=alt.binding_select(options=SEX_OPTIONS, labels=SEX_LABELS,
                                           name='Sex  '))
sketch_2_2 = alt.Chart(heat_df).transform_filter(
    'datum.decade == decade_22 && datum.sex_mode == sex_22 && datum.nrank <= n_rows'
).mark_rect().encode(
    x=alt.X('dpt:N', title='Department (ordered by total births; metropolitan + Corsica)',
            sort=dept_order, axis=alt.Axis(labels=False, ticks=False)),
    y=alt.Y('preusuel:N', title='Name',
            sort=alt.EncodingSortField(field='nrank', op='min', order='ascending')),
    color=alt.Color('rel:Q', scale=alt.Scale(scheme='magma'),
                    legend=alt.Legend(title=['Rate vs', 'row max'], format='.0%')),
    tooltip=[alt.Tooltip('preusuel:N', title='Name'), alt.Tooltip('sex_mode:N', title='Sex mode'),
             alt.Tooltip('decade:O', title='Decade'), alt.Tooltip('nom:N', title='Department'),
             alt.Tooltip('per_1000:Q', title='Births / 1000', format='.2f'),
             alt.Tooltip('rel:Q', title='vs row max', format='.0%')],
).add_params(decade_22, sex_22, n_rows).properties(
    width=820, height=360,
    title='2.2 - Regional profile of the selected-decade top names (row-relative; slide decade and sex)')

sketch_2_2.save('sketch_2_2_heatmap.png', ppi=120)
sketch_2_2


# Visualization 3 - Are there gender effects?


### Sketch 3.5 - Popularity among boys vs among girls (decade slider)

x = the name's share of all male births that decade (‰), y = its share of all
female births (‰), log axes, `y = x` parity line (= unisex). Decade slider
1900-2020 animates; size = total births (√ scale).


In [ ]:
g3 = (named.dropna(subset=['decade'])
      .groupby(['decade', 'preusuel', 'sexe'])['nombre'].sum()
      .unstack(fill_value=0).rename(columns={1: 'male', 2: 'female'}).reset_index())
tot3 = (named.dropna(subset=['decade']).groupby(['decade', 'sexe'])['nombre'].sum()
        .unstack(fill_value=0).rename(columns={1: 'tmale', 2: 'tfemale'}).reset_index())
g3 = g3.merge(tot3, on='decade')
g3 = g3[(g3.male > 0) & (g3.female > 0)].copy()
g3['male_pm'] = (1000 * g3.male / g3.tmale).round(4)
g3['female_pm'] = (1000 * g3.female / g3.tfemale).round(4)
g3['total'] = g3.male + g3.female
g3['male_share'] = (g3.male / g3.total).round(4)
g3['rank'] = g3.groupby('decade')['total'].rank(ascending=False, method='first')
g3['decade'] = g3['decade'].astype(int)

n_lab = alt.param(name='n_lab', value=8,
                  bind=alt.binding_range(min=0, max=200, step=1, name='Labelled names  '))
decade_sel = alt.selection_point(fields=['decade'], value=[{'decade': 2000}],
                                 bind=alt.binding_range(min=1900, max=2020, step=10, name='Decade  '))
lo = float(g3[['male_pm', 'female_pm']].min().min())
hi = float(g3[['male_pm', 'female_pm']].max().max())
parity = alt.Chart(pd.DataFrame({'v': [lo, hi]})).mark_line(
    color='#444', strokeDash=[6, 4]).encode(x='v:Q', y='v:Q')
pts = alt.Chart(g3).mark_circle(opacity=0.7, stroke='#888', strokeWidth=0.4).encode(
    x=alt.X('male_pm:Q', scale=alt.Scale(type='log'), title='Share of all BOYS (per 1000, log)'),
    y=alt.Y('female_pm:Q', scale=alt.Scale(type='log'), title='Share of all GIRLS (per 1000, log)'),
    size=alt.Size('total:Q', title='Total births', scale=alt.Scale(type='sqrt', range=[15, 1100])),
    color=alt.Color('male_share:Q', title='Male share', scale=alt.Scale(scheme='redblue', domain=[0, 1])),
    tooltip=[alt.Tooltip('preusuel:N', title='Name'), alt.Tooltip('decade:Q', title='Decade', format='d'),
             alt.Tooltip('male:Q', title='Boys', format=','), alt.Tooltip('female:Q', title='Girls', format=',')],
).transform_filter(decade_sel)
labels = alt.Chart(g3).mark_text(align='left', dx=6, fontSize=9).encode(
    x='male_pm:Q', y='female_pm:Q', text='preusuel:N').transform_filter(
    decade_sel).transform_filter('datum.rank <= n_lab')
sketch_3_5 = (parity + pts + labels).add_params(decade_sel, n_lab).properties(
    width=600, height=560,
    title='3.5 - Popularity among boys vs girls (parity line = unisex; drag the decade)')

sketch_3_5.save('sketch_3_5_scatter.png', ppi=120)
sketch_3_5

### Sketch 3.2 - Diverging gender bars by share (decade slider)

Each bar shows the name's gender share (full width), sorted from all-male to
all-female. The decade slider recomputes up to the top 300 names and their splits.
The rank selector chooses whether the list is ranked on mixed, boys, or girls births; watch DOMINIQUE turn unisex in the 1950s-60s.


In [ ]:
def gender_table(df, label, rank_mode):
    t = (df.groupby(['preusuel', 'sexe'])['nombre'].sum()
         .unstack(fill_value=0).rename(columns={1: 'male', 2: 'female'}))
    t['total'] = t['male'] + t['female']
    if rank_mode == 'Boys':
        t['rank_metric'] = t['male']
    elif rank_mode == 'Girls':
        t['rank_metric'] = t['female']
    else:
        t['rank_metric'] = t['total']
    t = t[t['rank_metric'] > 0].nlargest(300, 'rank_metric').reset_index()
    t['prank'] = range(1, len(t) + 1)
    t['male_share'] = (t['male'] / t['total']).round(4)
    t['decade'] = label
    t['rank_mode'] = rank_mode
    return t

tabs = []
for mode in SEX_OPTIONS:
    tabs.append(gender_table(named, 0, mode))
    for d in DECADES:
        tabs.append(gender_table(named[named.decade == d], d, mode))
g32 = pd.concat(tabs, ignore_index=True)
m = g32[['decade', 'rank_mode', 'preusuel', 'male', 'total', 'male_share', 'prank']].rename(columns={'male': 'n'})
m['sexe'] = 'Male'; m['signed'] = -(m['n'] / m['total'])
f = g32[['decade', 'rank_mode', 'preusuel', 'female', 'total', 'male_share', 'prank']].rename(columns={'female': 'n'})
f['sexe'] = 'Female'; f['signed'] = f['n'] / f['total']
long32 = pd.concat([m, f], ignore_index=True)
long32['signed'] = long32['signed'].round(4)

n_names = alt.param(name='n_names', value=20,
                    bind=alt.binding_range(min=10, max=300, step=1, name='Top N names  '))
decade_32 = alt.param(name='decade_32', value=2010,
                       bind=alt.binding_range(min=1900, max=2020, step=10,
                                              name='Decade  '))
rank_mode3 = alt.param(name='rank_mode3', value='Mixed',
                       bind=alt.binding_select(options=SEX_OPTIONS, labels=SEX_LABELS,
                                               name='Rank names by  '))
sketch_3_2 = alt.Chart(long32).transform_filter(
    'datum.decade == decade_32 && datum.rank_mode == rank_mode3 && datum.prank <= n_names'
).mark_bar().encode(
    x=alt.X('signed:Q', title='100% Male  <-  share of births  ->  100% Female',
            axis=alt.Axis(format='+%'), scale=alt.Scale(domain=[-1, 1])),
    y=alt.Y('preusuel:N', title='Name',
            sort=alt.EncodingSortField(field='male_share', op='min', order='descending')),
    color=alt.Color('sexe:N', title='Sex',
                    scale=alt.Scale(domain=['Male', 'Female'], range=['#4c78a8', '#e45756'])),
    tooltip=[alt.Tooltip('preusuel:N', title='Name'), alt.Tooltip('rank_mode:N', title='Ranked by'),
             alt.Tooltip('sexe:N'), alt.Tooltip('n:Q', title='Births', format=',')],
).add_params(decade_32, n_names, rank_mode3).properties(
    width=720, height=470,
    title='3.2 - Gender split of the decade top names (slide decade, N, and ranking sex)')

sketch_3_2.save('sketch_3_2_violin.png', ppi=120)
sketch_3_2


### Sketch 3.3 - Names by gender lean (year slider, all 121 years)

X is a logit of the male share (centre = unisex, Male on the left); colour is a
3-class gender label; size = births (√). Only genuinely co-ed names (each sex ≥ 5%)
are labelled. The year slider covers 1900-2020.


In [ ]:
ya = (named.dropna(subset=['year'])
      .groupby(['year', 'preusuel', 'sexe'])['nombre'].sum()
      .unstack(fill_value=0).rename(columns={1: 'male', 2: 'female'}).reset_index())
ya['total'] = ya['male'] + ya['female']
ya['rank'] = ya.groupby('year')['total'].rank(ascending=False, method='first')
ya = ya[ya['rank'] <= 1000].copy()
ya['male_share'] = ya['male'] / ya['total']
s = ya['male_share'].clip(0.01, 0.99)
ya['lean'] = -np.log(s / (1 - s))  # negative => male on the LEFT (sketch orientation)
ya['cls'] = np.where(ya.male_share >= 0.6, 'Mostly male',
                     np.where(ya.male_share <= 0.4, 'Mostly female', 'Unisex'))
ya['mixed'] = ya['male_share'].between(0.05, 0.95)  # given to both sexes >=5% -> co-ed (labelled)
ya['year'] = ya['year'].astype(int)

n_pts = alt.param(name='n_pts', value=300,
                  bind=alt.binding_range(min=50, max=1000, step=50, name='Top N names plotted  '))
year_sel = alt.selection_point(fields=['year'], value=[{'year': 2000}],
                               bind=alt.binding_range(min=1900, max=2020, step=1, name='Year  '))
pts = alt.Chart(ya).mark_circle(opacity=0.8, stroke='#777', strokeWidth=0.3).encode(
    x=alt.X('lean:Q', title='<- more MALE      gender lean      more FEMALE ->',
            axis=alt.Axis(labels=False, ticks=False, grid=False)),
    y=alt.Y('total:Q', title='Births this year (log)', scale=alt.Scale(type='log')),
    size=alt.Size('total:Q', scale=alt.Scale(type='sqrt', range=[20, 1100]), title='Births'),
    color=alt.Color('cls:N', title='Gender class',
                    scale=alt.Scale(domain=['Mostly male', 'Unisex', 'Mostly female'],
                                    range=['#4c78a8', '#f58518', '#e45756'])),
    tooltip=[alt.Tooltip('preusuel:N', title='Name'), alt.Tooltip('year:Q', title='Year', format='d'),
             alt.Tooltip('total:Q', title='Births', format=','),
             alt.Tooltip('male_share:Q', title='Male share', format='.0%')],
).transform_filter(year_sel).transform_filter('datum.rank <= n_pts')
mid = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(color='#888', strokeDash=[4, 4]).encode(x='x:Q')
txt = alt.Chart(ya).mark_text(dy=-12, fontSize=9, fontWeight='bold').encode(
    x='lean:Q', y='total:Q', text='preusuel:N').transform_filter(
    year_sel).transform_filter('datum.rank <= n_pts').transform_filter(alt.datum.mixed == True)
sketch_3_3 = (mid + pts + txt).add_params(year_sel, n_pts).properties(
    width=760, height=470,
    title=alt.TitleParams(
        text='3.3 - Names by gender lean and popularity (drag the year)',
        subtitle='Almost every French name is strongly gendered (two clusters); labelled names are the rare co-ed ones.',
        anchor='start'))

sketch_3_3.save('sketch_3_3_positional.png', ppi=120)
sketch_3_3